<a href="https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Example%20Final%20Project/ISBA2411_Final_Project_Example_Helios_Support_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Helios Support Copilot — An Agentic Customer-Support System
### ISBA 2411 · Natural Language Processing · Final Project (worked example)

**What this is.** A complete, end-to-end example of the kind of system your team will build for the final project. It walks the entire arc of the course — from a classical baseline to a transformer model to a retrieval-augmented LLM to a **multi-agent workflow** — on a real text dataset, and it ends with evaluation and a governance appendix, exactly as the rubric asks.

**The business problem (fictional company "Helios").** Helios is a mid-size electronics retailer. Its support inbox is overwhelmed: thousands of tickets a week, slow first responses, and agents answering the same questions over and over. We will build a copilot that **triages** each incoming ticket, **retrieves** the right help-center article, **drafts** a grounded reply, and **decides** whether the reply is safe to auto-send or should be **escalated to a human**.

**Why agents?** A single LLM call cannot safely run support. Real triage needs routing (different tickets go different ways), tool use (search the knowledge base), and a self-check before anything reaches a customer. That is an *agentic* workflow, and we build it with **LangGraph** — a small graph of specialized nodes with conditional control flow.

**How this maps to your milestones**

| Milestone | Section here |
|---|---|
| M1 · Business Brief | §1 |
| M2 · Baseline & Representation | §3 (TF-IDF + Logistic Regression) |
| M3 · Model Adaptation | §4 (transformer embeddings) + §5 (retrieval index) |
| M4 · System Prototype | §6 (single-shot RAG) |
| M5 · Final System & Demo | §7 (the agentic LangGraph system) |
| Evaluation + Governance | §8 + §9 |

---
**How to run this notebook**
1. `Runtime → Run all`. A GPU is optional (the embedding model is small enough for CPU).
2. The LLM sections need a free **Gemini API key** from [aistudio.google.com](https://aistudio.google.com/app/apikey). Add it to Colab Secrets as `GEMINI_API_KEY` (key icon in the left sidebar), or paste it when prompted.
3. The free tier is rate-limited (about 10 requests/minute), so the agent demo and evaluation run on a small sample with short pauses built in.

> **Text source / attribution.** Customer messages come from the **Bitext Customer Support LLM Chatbot Training Dataset** (`bitext/Bitext-customer-support-llm-chatbot-training-dataset`), released under CDLA-Sharing-1.0. The Helios help-center articles in §5 are original content written for this example.


## 0 · Setup

Install the stack, set configuration, and load the Gemini API key. Everything is free and runs in the browser.

In [ ]:
# Install dependencies (quiet). Takes ~1-2 minutes on a fresh Colab runtime.
%pip install -q "datasets>=3.0" "sentence-transformers>=3.0" "langgraph>=0.2" "langchain-google-genai>=2.0" "langchain-core>=0.3"
print("Dependencies installed.")

In [ ]:
import os, json, re, time, textwrap, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

# ---- Configuration -------------------------------------------------
# Free-tier-friendly Gemini model. Alternatives if needed: "gemini-2.5-flash-lite", "gemini-3-flash-preview".
LLM_MODEL   = "gemini-2.5-flash"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SAMPLE_SIZE = 4000          # rows sampled from the dataset for speed
RANDOM_SEED = 42
print(f"LLM model:       {LLM_MODEL}")
print(f"Embedding model: {EMBED_MODEL}")

In [ ]:
# ---- Gemini API key: Colab Secret -> env var -> prompt -------------
import getpass
def load_gemini_key():
    key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None
    if not key:
        try:
            key = getpass.getpass("Paste your Gemini API key (leave blank to skip LLM sections): ").strip()
        except Exception:
            key = None
    if key:
        os.environ["GOOGLE_API_KEY"] = key      # langchain-google-genai reads this
        os.environ["GEMINI_API_KEY"] = key
    return key

GEMINI_KEY    = load_gemini_key()
LLM_AVAILABLE = bool(GEMINI_KEY)
print("LLM available:" , LLM_AVAILABLE,
      "\\n(classical + embedding sections run regardless; RAG + agent sections need the key.)")

In [ ]:
# ---- One small LLM helper used everywhere --------------------------
llm = None
if LLM_AVAILABLE:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import SystemMessage, HumanMessage
    llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0.2, max_retries=3)

def call_llm(prompt, system=None, temperature=None):
    """Single text-in / text-out call. Returns '' if no key is configured."""
    if not LLM_AVAILABLE:
        return ""
    from langchain_core.messages import SystemMessage, HumanMessage
    msgs = []
    if system: msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    model = llm if temperature is None else ChatGoogleGenerativeAI(
        model=LLM_MODEL, temperature=temperature, max_retries=3)
    return model.invoke(msgs).content

def parse_json(text):
    """Robustly pull a JSON object out of an LLM response (strips code fences)."""
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m: text = m.group(0)
    try:
        return json.loads(text)
    except Exception:
        return {}

# quick smoke test
if LLM_AVAILABLE:
    print(call_llm("Reply with exactly the word: ready"))
else:
    print("Skipping LLM smoke test (no key).")

## 1 · Business Brief  (Milestone 1)

*This is the kind of brief your M1 deliverable should contain — problem, user, value, hypothesis, risks.*

**Problem.** Helios support receives ~5,000 tickets/week. Median first-response time is 14 hours; ~60% of tickets are repetitive, policy-answerable questions (order tracking, returns, refunds, password resets). Agents are expensive and burning out on low-complexity volume.

**Users.** (1) *Customers* who want a fast, correct answer. (2) *Support agents* who should spend their time on the hard 40%, not the easy 60%. (3) *Support managers* accountable for response time, CSAT, and compliance.

**Proposed solution.** An AI copilot that classifies each ticket, drafts a help-center-grounded reply, and either auto-resolves it or hands a summarized, pre-drafted ticket to an agent. A human stays in the loop for anything sensitive, low-confidence, or out-of-policy.

**Hypothesis.** A grounded, self-checking agent can safely auto-resolve a meaningful share of routine tickets while *escalating* everything risky, cutting first-response time without lowering answer quality.

**Primary risks (developed in §9).** Hallucinated policy, leaking or mishandling personal data, tone failures on angry customers, and silent model drift. The design mitigates each with grounding, a guardrail review step, and mandatory human escalation paths.

**Success metrics.** Auto-resolution rate, escalation precision (did we escalate the things we should?), groundedness of auto-sent replies, and — in production — first-response time and CSAT.


## 2 · The data  (text source + quick EDA)

We use real customer-service messages from the **Bitext Customer Support** dataset. Each row has a customer `instruction` (the message), a `category` and `intent` label, and a reference `response`. We will predict `category` as our triage label and reuse messages as test tickets for the agent.

In [ ]:
from datasets import load_dataset

def load_support_data():
    """Load the Bitext customer-support dataset; fall back to a tiny embedded sample if offline."""
    try:
        ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
        df = ds.to_pandas()
        cols = [c for c in ["instruction","category","intent","response"] if c in df.columns]
        return df[cols].dropna(subset=["instruction","category"]).reset_index(drop=True), "Bitext (Hugging Face)"
    except Exception as e:
        print("Could not load from Hugging Face (", e, ") — using a small embedded sample.")
        sample = [
            ("I want to track my order","ORDER","track_order"),
            ("how can i get a refund for my purchase","REFUND","get_refund"),
            ("i need to cancel order number 12345","ORDER","cancel_order"),
            ("I forgot my password and cannot log in","ACCOUNT","recover_password"),
            ("what payment methods do you accept","PAYMENT","check_payment_methods"),
            ("my package arrived damaged","DELIVERY","delivery_options"),
            ("how long does shipping take","DELIVERY","delivery_period"),
            ("I want to change my shipping address","SHIPPING","change_shipping_address"),
            ("where is my invoice","INVOICE","get_invoice"),
            ("I want to speak to a human agent","CONTACT","contact_human_agent"),
        ] * 30
        df = pd.DataFrame(sample, columns=["instruction","category","intent"])
        df["response"] = ""
        return df, "embedded fallback sample"

raw_df, source_name = load_support_data()
print(f"Loaded {len(raw_df):,} rows from: {source_name}")
raw_df.head(3)

In [ ]:
# Sample for speed and show the category distribution
df = raw_df.sample(min(SAMPLE_SIZE, len(raw_df)), random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Working sample: {len(df):,} rows · {df['category'].nunique()} categories\n")
print(df["category"].value_counts())

import matplotlib.pyplot as plt
ax = df["category"].value_counts().plot(kind="bar", figsize=(9,3.5), color="#B31A38")
ax.set_title("Ticket volume by category (sample)"); ax.set_ylabel("tickets"); plt.tight_layout(); plt.show()

## 3 · Baseline & Representation  (Milestone 2)

*Always start with an honest, cheap baseline that the fancy model has to beat.* Here: bag-of-words **TF-IDF** features + **Logistic Regression** to predict the ticket category. This is fast, interpretable, and connects directly to J&M Ch. 2-4.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

X = df["instruction"].astype(str)
y = df["category"].astype(str)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)),
    ("clf",   LogisticRegression(max_iter=2000)),
])
baseline.fit(X_tr, y_tr)
base_pred = baseline.predict(X_te)

base_acc = accuracy_score(y_te, base_pred)
base_f1  = f1_score(y_te, base_pred, average="macro")
print(f"TF-IDF + LogReg   accuracy={base_acc:.3f}   macro-F1={base_f1:.3f}\n")
print(classification_report(y_te, base_pred))

## 4 · Model adaptation: transformer embeddings  (Milestone 3)

Now we represent each message with a **sentence-transformer** (a small BERT-family encoder) instead of word counts, then train the same simple classifier on those dense vectors. The lesson is not "fancy always wins" — on templated text TF-IDF is a strong opponent — it is to *measure the trade* honestly: accuracy vs. cost vs. the ability to generalize to unseen phrasings. This connects to HOLLM Ch. 2-3 and Tunstall Ch. 4.

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL)

E_tr = embedder.encode(X_tr.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
E_te = embedder.encode(X_te.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)

emb_clf = LogisticRegression(max_iter=2000)
emb_clf.fit(E_tr, y_tr)
emb_pred = emb_clf.predict(E_te)

emb_acc = accuracy_score(y_te, emb_pred)
emb_f1  = f1_score(y_te, emb_pred, average="macro")
print(f"MiniLM + LogReg   accuracy={emb_acc:.3f}   macro-F1={emb_f1:.3f}")

In [ ]:
# Compare the two representations side by side
compare = pd.DataFrame({
    "model":   ["TF-IDF + LogReg", "MiniLM embeddings + LogReg"],
    "accuracy":[base_acc, emb_acc],
    "macro_F1":[base_f1, emb_f1],
})
display(compare)

ax = compare.set_index("model")[["accuracy","macro_F1"]].plot(
        kind="bar", figsize=(7,3.4), color=["#1E2A42","#B31A38"], rot=0)
ax.set_ylim(0,1); ax.set_title("Baseline vs. embeddings"); plt.tight_layout(); plt.show()

winner = "embeddings" if emb_f1 > base_f1 else "the TF-IDF baseline"
print(f"On this sample, {winner} scores higher on macro-F1. "
      "Report whichever wins — and note the baseline is the number to beat in your M2/M3.")

## 5 · The knowledge base + retrieval index  (Milestone 3 → 4)

A support copilot must answer *from the company's actual policies*, not from the model's memory. We build a small **help-center knowledge base** (original Helios content), embed it, and write a semantic `retrieve()` function. This is the retrieval half of RAG.

In [ ]:
# Helios help-center articles (original content for this example).
KB = [
 {"id":"KB01","title":"Returns policy",
  "text":"Most items can be returned within 30 days of delivery for a full refund, unused and in original packaging. Electronics have a 15-day return window. Refunds go back to the original payment method within 5-7 business days after we receive the item."},
 {"id":"KB02","title":"Track your order",
  "text":"To track an order, sign in and go to Account > Orders, then select the order to see live carrier tracking. A tracking link is also emailed when the order ships. Tracking can take up to 24 hours to update after dispatch."},
 {"id":"KB03","title":"Shipping options and times",
  "text":"Standard shipping is 3-5 business days and is free on orders over $50. Expedited shipping is 2 business days. Overnight is next business day if ordered before 2pm. International delivery is 7-14 business days."},
 {"id":"KB04","title":"Cancel an order",
  "text":"Orders can be cancelled within 60 minutes of being placed from Account > Orders > Cancel. After 60 minutes the order may already be in fulfillment; if it has shipped, use the returns process once it arrives."},
 {"id":"KB05","title":"Refund status",
  "text":"After we receive a returned item, refunds are processed within 5-7 business days. You can check status under Account > Orders > Refunds. Bank posting times vary by provider after we issue the refund."},
 {"id":"KB06","title":"Payment methods",
  "text":"Helios accepts major credit and debit cards, PayPal, and Helios gift cards. We do not accept checks or wire transfers. If a payment fails, verify the billing address and card expiry, then retry; repeated failures may require contacting your bank."},
 {"id":"KB07","title":"Reset your password",
  "text":"Use the Forgot Password link on the sign-in page. A reset link is emailed and is valid for 60 minutes. If it expires, request a new one. For security, Helios never asks for your password by email or phone."},
 {"id":"KB08","title":"Warranty and claims",
  "text":"Electronics include a 1-year limited warranty covering manufacturing defects. To file a claim, go to Account > Orders > Warranty and provide proof of purchase. Accidental or cosmetic damage is not covered."},
 {"id":"KB09","title":"Damaged or wrong item",
  "text":"Report a damaged or incorrect item within 48 hours of delivery via Account > Orders > Report a Problem, including photos. We will send a prepaid return label and ship a replacement or issue a refund."},
 {"id":"KB10","title":"Contacting support and security",
  "text":"Live chat and phone support are available 9am-7pm ET, Monday to Saturday. For your security, never share full card numbers, passwords, or one-time codes in a message; Helios staff will never ask for them."},
]
kb_emb = embedder.encode([a["text"] for a in KB], normalize_embeddings=True)

def retrieve(query, k=3):
    """Return the top-k KB articles by cosine similarity."""
    q = embedder.encode([query], normalize_embeddings=True)[0]
    sims = kb_emb @ q
    order = sims.argsort()[::-1][:k]
    return [{"id":KB[i]["id"], "title":KB[i]["title"], "text":KB[i]["text"], "score":float(sims[i])} for i in order]

# sanity check
for hit in retrieve("my package never showed up and I want my money back"):
    print(f"  {hit['id']} {hit['title']:28s} score={hit['score']:.2f}")

## 6 · Prototype: single-shot RAG  (Milestone 4)

The simplest LLM solution: retrieve relevant articles and ask Gemini to answer **using only that context**. This is a real prototype — but notice what it *cannot* do: it does not decide whether the question is in scope, it does not check its own answer, and it will happily reply to an angry or sensitive message. That motivates the agent in §7.

In [ ]:
SYSTEM = ("You are Helios Support, a careful customer-service assistant. "
          "Answer ONLY using the provided help-center context. If the context does not "
          "contain the answer, say you are not sure and offer to connect a human. "
          "Cite the article ids you used in square brackets, e.g. [KB02]. Be concise and friendly.")

def rag_answer(ticket, k=3):
    hits = retrieve(ticket, k=k)
    context = "\n\n".join(f"[{h['id']}] {h['title']}: {h['text']}" for h in hits)
    prompt = f"Help-center context:\n{context}\n\nCustomer message:\n{ticket}\n\nDraft a reply:"
    return rag_answer_text(prompt), hits

def rag_answer_text(prompt):
    return call_llm(prompt, system=SYSTEM, temperature=0.3)

if LLM_AVAILABLE:
    demo = "Hi, how long do I have to return a laptop, and how do refunds work?"
    ans, hits = rag_answer(demo)
    print("CUSTOMER:", demo, "\n")
    print("DRAFT:\n", ans)
else:
    print("Add a Gemini key to run the RAG prototype.")

## 7 · The agentic system  (Milestone 5)

This is the heart of the project. Instead of one call, we build a **LangGraph** workflow of specialized agents with conditional routing:

```
            ┌─────────┐
  ticket ─▶ │ triage  │  classify: category, urgency, sentiment, sensitive?
            └────┬────┘
                 │ route
        sensitive/high-urgency ──────────────▶ ESCALATE ─▶ human
                 │ otherwise
            ┌────▼────┐   ┌────────┐   ┌────────┐
            │retrieve │─▶ │ draft  │─▶ │ review │  grounded? on-policy?
            └─────────┘   └────────┘   └───┬────┘
                                  approve  │  reject
                              AUTO-SEND ◀──┴──▶ ESCALATE ─▶ human
```

Each node is a small, single-responsibility agent. The graph makes the **control flow explicit and auditable** — you can see exactly why any given ticket was auto-sent or escalated.

In [ ]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END

# Categories we never auto-resolve, regardless of confidence:
ESCALATE_CATEGORIES = {"complaint", "contact_human_agent", "legal", "other"}

class TicketState(TypedDict, total=False):
    ticket: str
    category: str
    urgency: str
    sentiment: str
    sensitive: bool
    route: str
    retrieved: List[dict]
    draft: str
    review_status: str
    grounded: bool
    reason: str
    final: str
    action: str

# ---- Node 1: triage (LLM classifies the ticket) --------------------
def triage_node(state: TicketState) -> dict:
    t = state["ticket"]
    prompt = (
        "Classify this customer-support message. Return ONLY JSON with keys: "
        "category (one short label), urgency (low|medium|high), "
        "sentiment (positive|neutral|negative), "
        "sensitive (true if it contains personal data like card numbers, threatens legal action, "
        "or is out of scope such as medical/legal advice).\n\n"
        f"Message: {t}"
    )
    data = parse_json(call_llm(prompt, system="You are a precise ticket triage classifier.", temperature=0.0))
    return {
        "category":  str(data.get("category","unknown")).lower(),
        "urgency":   str(data.get("urgency","medium")).lower(),
        "sentiment": str(data.get("sentiment","neutral")).lower(),
        "sensitive": bool(data.get("sensitive", False)),
    }

# ---- Router after triage -------------------------------------------
def route_after_triage(state: TicketState) -> str:
    if state.get("sensitive") or state.get("urgency") == "high" \
       or state.get("category") in ESCALATE_CATEGORIES:
        return "escalate"
    return "retrieve"

# ---- Node 2: retrieve (tool use) -----------------------------------
def retrieve_node(state: TicketState) -> dict:
    return {"retrieved": retrieve(state["ticket"], k=3)}

# ---- Node 3: draft (grounded generation) ---------------------------
def draft_node(state: TicketState) -> dict:
    hits = state["retrieved"]
    context = "\n\n".join(f"[{h['id']}] {h['title']}: {h['text']}" for h in hits)
    prompt = (f"Help-center context:\n{context}\n\nCustomer message:\n{state['ticket']}\n\n"
              "Draft a concise, friendly reply using ONLY the context. Cite article ids like [KB02].")
    return {"draft": call_llm(prompt, system=SYSTEM, temperature=0.3)}

# ---- Node 4: review (self-check / guardrail) -----------------------
def review_node(state: TicketState) -> dict:
    prompt = (
        "You are a compliance reviewer. Given the help-center context and a drafted reply, decide if the "
        "reply is (a) grounded ONLY in the context and (b) safe and on-policy. Return ONLY JSON with keys: "
        "grounded (true/false), status (APPROVE or ESCALATE), reason (one short sentence).\n\n"
        f"Context ids: {[h['id'] for h in state['retrieved']]}\n"
        f"Context:\n" + "\n".join(f"[{h['id']}] {h['text']}" for h in state['retrieved']) +
        f"\n\nDrafted reply:\n{state['draft']}"
    )
    data = parse_json(call_llm(prompt, system="You are a strict but fair compliance reviewer.", temperature=0.0))
    return {
        "grounded":      bool(data.get("grounded", False)),
        "review_status": str(data.get("status","ESCALATE")).upper(),
        "reason":        str(data.get("reason","")),
    }

# ---- Router after review -------------------------------------------
def route_after_review(state: TicketState) -> str:
    return "send" if state.get("review_status") == "APPROVE" and state.get("grounded") else "escalate"

# ---- Terminal nodes ------------------------------------------------
def send_node(state: TicketState) -> dict:
    return {"action": "AUTO_SEND", "final": state["draft"]}

def escalate_node(state: TicketState) -> dict:
    reason = state.get("reason") or (
        "sensitive content" if state.get("sensitive") else
        "high urgency" if state.get("urgency")=="high" else
        f"category '{state.get('category')}' requires a human")
    summary = (f"[ESCALATED] {reason}. "
               f"Triage: category={state.get('category')}, urgency={state.get('urgency')}, "
               f"sentiment={state.get('sentiment')}.")
    return {"action": "ESCALATE_TO_HUMAN", "final": summary, "reason": reason}

print("Nodes defined.")

In [ ]:
# ---- Wire the graph ------------------------------------------------
g = StateGraph(TicketState)
g.add_node("triage",   triage_node)
g.add_node("retrieve", retrieve_node)
g.add_node("draft",    draft_node)
g.add_node("review",   review_node)
g.add_node("send",     send_node)
g.add_node("escalate", escalate_node)

g.add_edge(START, "triage")
g.add_conditional_edges("triage", route_after_triage, {"retrieve":"retrieve", "escalate":"escalate"})
g.add_edge("retrieve", "draft")
g.add_edge("draft", "review")
g.add_conditional_edges("review", route_after_review, {"send":"send", "escalate":"escalate"})
g.add_edge("send", END)
g.add_edge("escalate", END)

app = g.compile()

# Show the graph structure (mermaid text always works; PNG if the renderer is available)
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print(app.get_graph().draw_mermaid())

In [ ]:
# ---- Run the agent on a few tickets --------------------------------
def run_agent(ticket, verbose=True):
    state = app.invoke({"ticket": ticket})
    if verbose:
        print("="*78)
        print("TICKET   :", ticket)
        print(f"TRIAGE   : category={state.get('category')} · urgency={state.get('urgency')} "
              f"· sentiment={state.get('sentiment')} · sensitive={state.get('sensitive')}")
        if state.get("retrieved"):
            print("RETRIEVED:", ", ".join(h["id"] for h in state["retrieved"]))
        print(f"ACTION   : {state.get('action')}")
        print("OUTPUT   :")
        print(textwrap.indent(textwrap.fill(state.get("final",""), 92), "           "))
    return state

if LLM_AVAILABLE:
    demo_tickets = [
        "How long do I have to return a laptop and when will I get my money back?",
        "I placed order 88231 ten minutes ago and need to cancel it.",
        "This is the THIRD broken item you sent me. I am contacting my lawyer.",          # -> escalate (legal)
        "Please charge my card 4111 1111 1111 1111 for the order.",                       # -> escalate (PII/sensitive)
    ]
    for t in demo_tickets:
        run_agent(t)
        time.sleep(7)   # stay under the free-tier rate limit
else:
    print("Add a Gemini key to run the agent.")

## 8 · Evaluation

Two layers, mirroring the rubric:

1. **Component metrics** — the triage classifier (we already have F1 from §3-4).
2. **System metrics** — run a labelled set of tickets through the agent and measure *auto-resolution rate*, *escalation rate*, and whether auto-sent replies are *grounded* (cite a KB id). We hand-label whether each ticket *should* auto-resolve so we can check the routing.

In [ ]:
# A small labelled eval set: (ticket, should_auto_resolve?)
eval_set = [
    ("Where is my order? It said delivered but I never got it.", True),
    ("What payment methods do you accept?",                       True),
    ("I forgot my password, how do I reset it?",                  True),
    ("I want to return a phone I bought 3 weeks ago.",            True),
    ("I am furious, this is the worst service ever, I will sue.", False),   # legal/angry -> escalate
    ("Can you diagnose why my chest hurts?",                      False),   # out of scope -> escalate
]

def evaluate(eval_set):
    rows = []
    for ticket, should_auto in eval_set:
        st = app.invoke({"ticket": ticket})
        action   = st.get("action")
        auto     = action == "AUTO_SEND"
        grounded = bool(re.search(r"\[KB\d+\]", st.get("final",""))) if auto else None
        rows.append({
            "ticket": ticket[:42] + ("…" if len(ticket)>42 else ""),
            "category": st.get("category"),
            "action": action,
            "should_auto": should_auto,
            "routed_correctly": auto == should_auto,
            "grounded": grounded,
        })
        time.sleep(7)   # rate limit
    return pd.DataFrame(rows)

if LLM_AVAILABLE:
    results = evaluate(eval_set)
    display(results)
    auto_rate  = (results["action"]=="AUTO_SEND").mean()
    esc_rate   = (results["action"]=="ESCALATE_TO_HUMAN").mean()
    route_acc  = results["routed_correctly"].mean()
    grounded_r = results.loc[results["action"]=="AUTO_SEND","grounded"].mean()
    print(f"\nAuto-resolution rate : {auto_rate:.0%}")
    print(f"Escalation rate      : {esc_rate:.0%}")
    print(f"Routing accuracy     : {route_acc:.0%}   (did we auto/escalate the right tickets?)")
    print(f"Grounded auto-replies: {grounded_r:.0%}  (auto-sent replies that cite a KB article)")
    print(f"\nComponent recap — triage classifier macro-F1: "
          f"baseline {base_f1:.3f} vs embeddings {emb_f1:.3f}")
else:
    print("Add a Gemini key to run the evaluation.")

> **Reading the numbers.** This is a tiny demo set, so treat the percentages as illustrative. The point is the *shape* of an evaluation: separate the component (classifier F1) from the system (routing accuracy, groundedness), and always check that the guardrail escalates the things it should. In your project, scale the eval set to dozens of labelled tickets per category and add an LLM-as-judge or human rating of reply quality.

## 9 · Governance & Risk appendix

*Required in the final deliverable. Tie each risk to a concrete mitigation already in the system.*

**Data & licensing.** Training/eval messages come from the Bitext Customer Support dataset (CDLA-Sharing-1.0); derivatives must keep the same license and attribution. The Helios KB is original. No real customer data is used here; in production, ticket data is personal data and must be access-controlled, retention-limited, and excluded from third-party training.

**Privacy / PII.** Customers paste card numbers, addresses, and order IDs. The triage node flags `sensitive=true` for PII and routes straight to a human; the system never acts on payment details, and KB10 encodes the rule that staff never request secrets. Production should add automated PII redaction before any text reaches the LLM and log only redacted text.

**Hallucination.** Mitigated in three layers: (1) generation is constrained to retrieved context, (2) the reviewer node checks groundedness and rejects ungrounded drafts, (3) anything not approved is escalated rather than sent. Auto-sent replies must cite a KB id.

**Bias & fairness.** The classifier can underperform on colloquial or non-native phrasing (the dataset even tags these variations). Report per-category and per-style F1, not just the average, and monitor escalation rates across segments so one group is not systematically under-served.

**Human-in-the-loop.** Complaints, legal threats, high-urgency, low-confidence, and out-of-scope tickets are always escalated with a triage summary. The system assists agents; it does not replace the judgment path.

**Monitoring & drift.** In production, track auto-resolution rate, escalation precision, groundedness, and CSAT over time; alert when category mix or model confidence shifts (new products, policy changes). Re-evaluate after any KB or model update.

**Build vs. buy & cost.** Embeddings + classifier are cheap and run locally; LLM calls dominate cost and latency. Flash-tier models keep per-ticket cost low; cache frequent answers and only call the LLM when retrieval confidence is high. Reassess buy-vs-build as vendor copilots mature.

**Accountability.** Every decision is logged with its triage labels, retrieved ids, and review reason, so any auto-sent or escalated ticket can be audited after the fact.


## 10 · How to make this *your* project

This example is deliberately a strong B+. To push it further (and to make it yours), pick a few:

- **Swap the domain.** Bitext ships parallel datasets for telco, banking, insurance, e-commerce — change one string in §2 and re-run.
- **Strengthen retrieval.** Chunk longer KB articles, add a re-ranker, or report retrieval hit-rate against labelled (ticket → correct article) pairs.
- **Add a real tool.** Give the agent an `order_lookup(order_id)` function so it can answer "where is order 88231?" with live data, not just policy.
- **Add memory / multi-turn.** Let the graph carry conversation history so follow-up messages keep context (LangGraph state makes this natural).
- **Harden evaluation.** Scale to a labelled test set, add an LLM-as-judge for reply quality, and run an adversarial / red-team set (prompt-injection in a ticket, jailbreak attempts) for the responsible-AI section.
- **Measure the human edge.** Compare auto-sent replies to agent-written ones on a sample and report where the human still wins — that analysis is exactly the kind of judgment your final presentation should show.

**Deliverable mapping recap:** §1 → M1, §3 → M2, §4-5 → M3, §6 → M4, §7 → M5, §8-9 → evaluation + governance.
